# Week 20 Environment Smoke Test

Run this from the Databricks course cluster as a test student BEFORE class.
It proves the full Week 20 chain works end to end in the datacouch / us-west-2
account:

0. Per-student secret-scope auth + STS
1. Spark query of the Unity Catalog fraud table
2. Bedrock Converse (Sonnet 4.5)
3. Databricks-native MLflow
4. SageMaker control-plane calls
5. A real remote SageMaker training job
6. Register the model and deploy an endpoint
7. Invoke the endpoint
8. Tear the endpoint down

Each section prints PASS or raises loudly. Total wall time is about 30-40 min;
the training job and endpoint spin-up dominate.

There is NO Bedrock Knowledge Base step: the only KB that exists
(`bread-academy-fraud-policies`) lives in the di-mfa / us-east-1 account and is
unreachable from datacouch us-west-2. Week 20 does no real KB retrieval, so the
KB is intentionally out of scope here.

In [ ]:
# Section 0 - Per-student secret-scope auth and STS identity.
#
# Per-student AWS keys live in aws-course-creds-NN (NN derived from the
# Databricks username). Class-wide config lives in aws-course-shared.

import os
import json
import time
import boto3

_user = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().userName().get()
)
_num = _user.split("@")[0].split("-")[1] if _user.startswith("student-") else "01"
creds_scope = f"aws-course-creds-{_num}"
print("Databricks user:", _user, "-> creds scope:", creds_scope)

AWS_ACCESS_KEY_ID = dbutils.secrets.get(scope=creds_scope, key="aws-access-key-id")
AWS_SECRET_ACCESS_KEY = dbutils.secrets.get(scope=creds_scope, key="aws-secret-access-key")
AWS_REGION = dbutils.secrets.get(scope="aws-course-shared", key="aws-region")

os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_REGION"] = AWS_REGION
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION

sts = boto3.client("sts", region_name=AWS_REGION)
identity = sts.get_caller_identity()
print("STS caller ARN:", identity["Arn"])
assert identity["Account"] == "962804699607", "Not the datacouch account"
print("Section 0 PASS - auth and STS OK")

In [ ]:
# Section 1 - Spark query of the Unity Catalog fraud table.
#
# Confirms the cluster can read course_data and the synthetic table is loaded.

SOURCE_TABLE = "bread_academy.course_data.fraud_transactions"

row_count = spark.sql(f"SELECT COUNT(*) AS c FROM {SOURCE_TABLE}").collect()[0]["c"]
print(f"{SOURCE_TABLE}: {row_count:,} rows")
assert row_count > 1000, f"Expected ~45k rows, got {row_count}"

# Print the actual schema so column-name assumptions are visible.
cols = [f.name for f in spark.table(SOURCE_TABLE).schema.fields]
print("Columns:", cols)
for needed in ["narrative", "is_fraud", "amount", "partition_date"]:
    assert needed in cols, f"Column '{needed}' missing from {SOURCE_TABLE}"

print("Section 1 PASS - Unity Catalog and fraud table OK")


In [ ]:
# Section 2 - Bedrock Converse with Sonnet 4.5.
#
# Confirms the Databricks VNet can reach Bedrock and the datacouch account
# has model access for the cross-region Sonnet 4.5 inference profile.

BEDROCK_MODEL_ID = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)
resp = bedrock_runtime.converse(
    modelId=BEDROCK_MODEL_ID,
    messages=[{"role": "user", "content": [{"text": "Reply with the single word: ready"}]}],
    inferenceConfig={"maxTokens": 10, "temperature": 0},
)
reply = resp["output"]["message"]["content"][0]["text"]
print("Bedrock reply:", reply)
assert reply.strip(), "Empty Bedrock response"

print("Section 2 PASS - Bedrock Converse OK")


In [ ]:
# Section 3 - Databricks-native MLflow.
#
# The course uses built-in Databricks MLflow, not the SageMaker MLflow server.
# Each student logs to their own /Users/ experiment - auto-isolated, no setup.

import mlflow

mlflow.set_tracking_uri("databricks")
exp_path = f"/Users/{_user}/week20-env-smoke"
mlflow.set_experiment(exp_path)
print("MLflow experiment:", exp_path)

with mlflow.start_run(run_name=f"smoke-{int(time.time())}") as run:
    mlflow.log_param("smoke", "week20-env")
    mlflow.log_metric("ok", 1.0)
    run_id = run.info.run_id
print("Logged MLflow run:", run_id)

found = mlflow.search_runs(experiment_names=[exp_path])
assert len(found) >= 1, "MLflow run not found after logging"

print("Section 3 PASS - Databricks-native MLflow OK")


In [ ]:
# Section 4 - SageMaker control-plane calls.
#
# Confirms the per-student keys can talk to the SageMaker API and that
# iam:PassRole to the execution role is permitted.

sagemaker_client = boto3.client("sagemaker", region_name=AWS_REGION)

# Read-only: list training jobs.
jobs = sagemaker_client.list_training_jobs(MaxResults=5)
print("Recent training jobs visible:", len(jobs["TrainingJobSummaries"]))

# Class-wide config from the shared scope.
SAGEMAKER_ROLE_ARN = dbutils.secrets.get(
    scope="aws-course-shared", key="sagemaker-execution-role-arn"
)
S3_BUCKET = dbutils.secrets.get(scope="aws-course-shared", key="course-s3-bucket")
print("Execution role:", SAGEMAKER_ROLE_ARN)
print("Course S3 bucket:", S3_BUCKET)

# Idempotent control-plane write: create a model package group for the smoke.
# CreateModelPackageGroup raises a generic ValidationException (not a typed
# ResourceInUse) when the group already exists.
MPG_NAME = "week20-smoke-fraud-classifier"
try:
    sagemaker_client.create_model_package_group(
        ModelPackageGroupName=MPG_NAME,
        ModelPackageGroupDescription="Week 20 environment smoke test",
    )
    print("Created model package group:", MPG_NAME)
except sagemaker_client.exceptions.ClientError as e:
    if "already exists" in str(e):
        print("Model package group already exists:", MPG_NAME)
    else:
        raise

print("Section 4 PASS - SageMaker control-plane OK")


In [ ]:
# Section 5a - Export training data to S3.
#
# Take narrative + is_fraud from the fraud table, balance the classes a bit so
# DistilBERT has signal, and write a single CSV to the course bucket.

import pandas as pd

SMOKE_PREFIX = "week20-smoke"
TRAIN_S3_URI = f"s3://{S3_BUCKET}/{SMOKE_PREFIX}/train/train.csv"

train_pdf = (
    spark.table(SOURCE_TABLE)
    .select("narrative", "is_fraud")
    .toPandas()
    .rename(columns={"is_fraud": "label"})
)

# Down-sample the majority class so the smoke trains on a balanced ~4k rows.
fraud = train_pdf[train_pdf["label"] == 1]
legit = train_pdf[train_pdf["label"] == 0].sample(
    n=min(len(fraud) * 3, len(train_pdf[train_pdf["label"] == 0])), random_state=42
)
sample = pd.concat([fraud, legit]).sample(frac=1.0, random_state=42).reset_index(drop=True)
print(f"Training sample: {len(sample):,} rows")
print(sample["label"].value_counts().to_string())

# Write to S3 via boto3 (no Spark S3 write needed for a small CSV).
csv_bytes = sample.to_csv(index=False).encode("utf-8")
s3 = boto3.client("s3", region_name=AWS_REGION)
s3.put_object(
    Bucket=S3_BUCKET,
    Key=f"{SMOKE_PREFIX}/train/train.csv",
    Body=csv_bytes,
)
print("Uploaded training data:", TRAIN_S3_URI)
print("Section 5a PASS - training data in S3")


In [ ]:
# Section 5b - Write the training script to the cluster driver.
#
# The HuggingFace estimator needs a source_dir containing train.py. We write
# it inline so this notebook is fully self-contained on Databricks. This is
# the corrected script: it tokenizes the text before training (the earlier
# smoke jobs failed because raw text was passed to the Trainer untokenized).

import os
import textwrap

SOURCE_DIR = "/tmp/week20_smoke_src"
os.makedirs(SOURCE_DIR, exist_ok=True)

TRAIN_SCRIPT = textwrap.dedent('''\
    import argparse, os
    import numpy as np
    import pandas as pd
    from datasets import Dataset
    from transformers import (
        AutoModelForSequenceClassification, AutoTokenizer,
        Trainer, TrainingArguments,
    )

    def parse_args():
        p = argparse.ArgumentParser()
        p.add_argument("--epochs", type=int, default=1)
        p.add_argument("--batch_size", type=int, default=16)
        p.add_argument("--lr", type=float, default=2e-5)
        p.add_argument("--max_len", type=int, default=128)
        p.add_argument("--model_name", type=str, default="distilbert-base-uncased")
        p.add_argument("--num_labels", type=int, default=2)
        p.add_argument("--seed", type=int, default=42)
        p.add_argument("--train", type=str, default=os.environ.get("SM_CHANNEL_TRAIN"))
        p.add_argument("--model_dir", type=str,
                       default=os.environ.get("SM_MODEL_DIR", "/opt/ml/model"))
        return p.parse_args()

    def load_split(channel_dir):
        csvs = [os.path.join(channel_dir, f) for f in os.listdir(channel_dir)
                if f.endswith(".csv")]
        if not csvs:
            raise FileNotFoundError(f"No CSV in {channel_dir}")
        df = pd.concat([pd.read_csv(c) for c in csvs], ignore_index=True)
        text_col = "narrative" if "narrative" in df.columns else "text"
        df = df[[text_col, "label"]].rename(columns={text_col: "text"})
        df = df.dropna(subset=["text", "label"])
        df["label"] = df["label"].astype(int)
        return df

    def main():
        args = parse_args()
        np.random.seed(args.seed)
        df = load_split(args.train)
        print(f"Loaded {len(df):,} rows. Balance:")
        print(df["label"].value_counts().to_string())

        tokenizer = AutoTokenizer.from_pretrained(args.model_name)
        def tokenize(batch):
            return tokenizer(batch["text"], padding="max_length",
                             truncation=True, max_length=args.max_len)

        ds = Dataset.from_pandas(df, preserve_index=False)
        ds = ds.map(tokenize, batched=True)
        ds = ds.remove_columns(["text"])
        ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

        model = AutoModelForSequenceClassification.from_pretrained(
            args.model_name, num_labels=args.num_labels)
        targs = TrainingArguments(
            output_dir="/opt/ml/output", num_train_epochs=args.epochs,
            per_device_train_batch_size=args.batch_size, learning_rate=args.lr,
            logging_steps=50, save_strategy="no", seed=args.seed, report_to=[])
        Trainer(model=model, args=targs, train_dataset=ds).train()

        model.save_pretrained(args.model_dir)
        tokenizer.save_pretrained(args.model_dir)
        print(f"Saved model + tokenizer to {args.model_dir}")

    if __name__ == "__main__":
        main()
''')

with open(os.path.join(SOURCE_DIR, "train.py"), "w") as f:
    f.write(TRAIN_SCRIPT)
print("Wrote train.py to", SOURCE_DIR)


In [ ]:
# Section 5c - Launch the remote SageMaker training job.
#
# Real HuggingFace estimator, real g4dn.xlarge GPU instance. Proves the full
# training path from Databricks: PassRole, ECR image pull, S3 channel read,
# model.tar.gz written back to S3. Wall time roughly 8-12 min.
#
# HF 4.49.0 is the newest version sagemaker==2.257.3 supports for BOTH
# training and inference. Training DLC for HF 4.49.0 is pytorch 2.5.1 / py311.

import sagemaker
from sagemaker.huggingface import HuggingFace

sm_session = sagemaker.Session(boto_session=boto3.Session(region_name=AWS_REGION))
print("sagemaker SDK version:", sagemaker.__version__)

training_job_name = f"week20-smoke-train-{int(time.time())}"

estimator = HuggingFace(
    entry_point="train.py",
    source_dir=SOURCE_DIR,
    role=SAGEMAKER_ROLE_ARN,
    instance_type="ml.g4dn.xlarge",
    instance_count=1,
    transformers_version="4.49.0",
    pytorch_version="2.5.1",
    py_version="py311",
    hyperparameters={
        "epochs": 1,
        "batch_size": 16,
        "model_name": "distilbert-base-uncased",
        "num_labels": 2,
    },
    output_path=f"s3://{S3_BUCKET}/{SMOKE_PREFIX}/model-output",
    sagemaker_session=sm_session,
    base_job_name="week20-smoke-train",
)

# Blocking call - waits for the job to finish.
estimator.fit({"train": TRAIN_S3_URI}, job_name=training_job_name, wait=True)

trained_model_data = estimator.model_data
print("Training complete. Model artifact:", trained_model_data)
print("Section 5 PASS - remote training job succeeded")


In [ ]:
# Section 6 - Deploy the trained model to a real endpoint.
#
# Deploys the model trained in Section 5 behind the endpoint name Week 20
# expects: fraud-classifier-endpoint. CPU instance for inference. Wall time
# roughly 6-10 min for the endpoint to reach InService.
#
# Inference DLC for HF 4.49.0 is pytorch 2.6.0 / py312 - a different base
# from the training DLC, which is expected for HuggingFace containers.

from sagemaker.huggingface import HuggingFaceModel

ENDPOINT_NAME = "fraud-classifier-endpoint"

hf_model = HuggingFaceModel(
    model_data=trained_model_data,
    role=SAGEMAKER_ROLE_ARN,
    transformers_version="4.49.0",
    pytorch_version="2.6.0",
    py_version="py312",
    sagemaker_session=sm_session,
)

# If a prior endpoint with this name exists, delete its endpoint object first
# so deploy can recreate it cleanly. (Config/model are left in place.)
try:
    sagemaker_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
    print("Existing endpoint found - deleting before redeploy:", ENDPOINT_NAME)
    sagemaker_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
    waiter = sagemaker_client.get_waiter("endpoint_deleted")
    waiter.wait(EndpointName=ENDPOINT_NAME)
except sagemaker_client.exceptions.ClientError:
    print("No existing endpoint - fresh deploy.")

predictor = hf_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
    endpoint_name=ENDPOINT_NAME,
)
print("Endpoint deployed:", ENDPOINT_NAME)

status = sagemaker_client.describe_endpoint(EndpointName=ENDPOINT_NAME)["EndpointStatus"]
assert status == "InService", f"Endpoint status is {status}"
print("Section 6 PASS - endpoint InService")


In [ ]:
# Section 7 - Invoke the endpoint.
#
# Sends two narratives through the deployed endpoint and confirms a
# prediction comes back. The HuggingFace inference container expects
# {"inputs": "<text>"} and returns a list of label/score dicts.

sm_runtime = boto3.client("sagemaker-runtime", region_name=AWS_REGION)

samples = [
    "Customer cust_00037 swiped a $4213.20 wire_transfer at 02:17 UTC to a "
    "merchant in BR after 92 days of inactivity",
    "Customer cust_00102 made a $42.10 groceries purchase at 18:30 UTC at a "
    "local merchant",
]

for text in samples:
    resp = sm_runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps({"inputs": text}),
    )
    prediction = json.loads(resp["Body"].read())
    print("Input :", text[:70], "...")
    print("Output:", prediction)
    assert prediction, "Empty prediction from endpoint"

print("Section 7 PASS - endpoint invocation OK")


## Smoke test complete

If every section above printed PASS, the Week 20 environment is proven end to end:

| Section | Proves |
|---------|--------|
| 0 | Per-student secret scopes + STS reach the datacouch account |
| 1 | Unity Catalog and the synthetic fraud table are queryable |
| 2 | Bedrock Converse (Sonnet 4.5) is reachable with model access |
| 3 | Databricks-native MLflow logs and reads back runs |
| 4 | SageMaker control-plane calls and PassRole work |
| 5 | A remote SageMaker training job runs to completion |
| 6 | The trained model deploys to `fraud-classifier-endpoint` |
| 7 | The endpoint serves real predictions |

### Left running on purpose

`fraud-classifier-endpoint` is still InService - Week 20 reuses it, and we
are not tearing anything down yet. It bills at roughly $0.23/hr on
`ml.m5.xlarge`. Delete it after Week 20 if it is no longer needed.

### Notes for the Week 20 notebooks (separate fix, not done here)

The smoke test surfaced bugs in the Week 20 course notebooks themselves:

- They read secret scope `aws-course-creds` - the real scopes are
  `aws-course-creds-NN` (per student) and `aws-course-shared`.
- The data-engineering notebook reads keys `aws-access-key` / `aws-secret-key`
  - the real keys are `aws-access-key-id` / `aws-secret-access-key`.
- The main notebook hardcodes `KB_ID = "FARSQGTONR"` and probes a Bedrock KB.
  No KB exists in datacouch us-west-2; that probe should be removed.
- Column references `transaction_amount`, `hours_since_last_txn`,
  `transaction_date` do not exist. Real columns: `amount`,
  `days_since_last_txn`, `partition_date`.
- The main notebook does `%run ./week19_supervisor_helper`, which is not in
  the workspace.

Fix those in the Week 20 plan files, then regenerate the notebooks.
